# XGBoost

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
assets = ['BTC', 'ETH', 'XRP', 'SOL']

feature_cols = [
    'open', 'high', 'low', 'close', 'volume', 'quote_asset_volume', 'number_of_trades',
    'ret_1h_close', 'log_ret_1h_close', 'ret_3h', 'log_ret_3h',
    'ret_6h', 'log_ret_6h', 'ret_12h', 'log_ret_12h',
    'range_pct', 'body_pct', 'upper_shadow_pct', 'lower_shadow_pct', 'close_loc',
    'close_to_sma_12', 'close_to_sma_24',
    'close_to_ema_12', 'close_to_ema_24',
    'vol_24h', 'atr_14_pct', 'rsi_14',
    'macd_line_pct', 'macd_signal_pct', 'macd_hist_pct',
    'log_volume', 'log_quote_volume', 'log_trades',
    'vol_z_24', 'quote_vol_z_24', 'trades_z_24',
    'taker_buy_base_share', 'taker_buy_quote_share', 'order_flow_imbalance',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos'
]

target_col = 'label_up'

train_data, val_data, test_data = {}, {}, {}

for asset in assets:
    train_data[asset] = pd.read_csv(f"data/split_data/{asset}_train.csv")
    val_data[asset]   = pd.read_csv(f"data/split_data/{asset}_val.csv")
    test_data[asset]  = pd.read_csv(f"data/split_data/{asset}_test.csv")

## XGBoost

In [ ]:
def run_xgboost(train_df, val_df, asset_name, feature_cols, target_col):
    usable_features = [col for col in feature_cols if col in train_df.columns and col in val_df.columns]

    train_df = train_df.dropna(subset=usable_features + [target_col]).copy()
    val_df = val_df.dropna(subset=usable_features + [target_col]).copy()

    X_train = train_df[usable_features]
    y_train = train_df[target_col]
    X_val = val_df[usable_features]
    y_val = val_df[target_col]

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    acc = accuracy_score(y_val, y_pred)

    print(f"\n--- XGBoost: {asset_name} ---")
    print(f"Validation Accuracy: {acc:.4f}")
    print(confusion_matrix(y_val, y_pred))
    print(classification_report(y_val, y_pred))

    return {
        'model': model,
        'features_used': usable_features,
        'accuracy': acc,
        'y_val': y_val,
        'y_pred': y_pred,
        'y_prob': y_prob
    }

In [ ]:
results = {}
for asset in assets:
    results[asset] = run_xgboost(
        train_data[asset],
        val_data[asset],
        asset,
        feature_cols,
        target_col
    )

## Tune Probability Threshold Using Sharpe Ratio

In [ ]:
def tune_threshold_sharpe(y_prob, val_df, thresholds, cost=0.00035):

    best_sharpe = -np.inf
    best_threshold = None

    sharpe_list = []

    for t in thresholds:
        signal = (y_prob >= t).astype(int)

        # strategy returns
        ret = signal * val_df['fwd_ret_open_1h']

        # transaction cost
        turnover = np.abs(np.diff(signal, prepend=0))
        ret = ret - turnover * cost

        # avoid division by zero
        if ret.std() == 0:
            sharpe = 0
        else:
            sharpe = ret.mean() / ret.std() * np.sqrt(8760)

        sharpe_list.append((t, sharpe))

        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_threshold = t

    return best_threshold, best_sharpe, sharpe_list

In [ ]:
threshold_grid = np.arange(0.45, 0.65, 0.01)

threshold_results = {}

for asset in assets:
    y_prob = results[asset]['y_prob']
    val_df = val_data[asset].loc[results[asset]['y_val'].index]

    best_t, best_sharpe, sharpe_curve = tune_threshold_sharpe(
        y_prob, val_df, threshold_grid
    )

    print(f"\n{asset}")
    print(f"Best threshold: {best_t}")
    print(f"Best Sharpe: {best_sharpe:.4f}")

    threshold_results[asset] = {
        'best_threshold': best_t,
        'best_sharpe': best_sharpe,
        'curve': sharpe_curve
    }

In [ ]:
for asset in assets:
    print(asset, threshold_results[asset]['best_threshold'])

The decision threshold is calibrated on the validation set by maximising the annualised Sharpe ratio under realistic transaction costs. This ensures that model outputs are translated into economically meaningful trading signals rather than purely statistical classifications.

## Tune Hyperparameters

In [ ]:
def tune_xgboost_and_threshold(train_df, val_df, asset, feature_cols, target_col):

    param_grid = {
        'max_depth': [3, 4, 5],
        'learning_rate': [0.05, 0.1],
        'n_estimators': [200, 300],
        'colsample_bytree': [0.7, 1.0]
    }
    threshold_grid = np.arange(0.45, 0.65, 0.01)

    best_result = {
        'max_depth': None,
        'learning_rate': None,
        'n_estimators': None,
        'colsample_bytree': None,
        'threshold': None,
        'sharpe': -np.inf
    }

    usable_features = [col for col in feature_cols if col in train_df.columns and col in val_df.columns]

    train_df = train_df.dropna(subset=usable_features + [target_col]).copy()
    val_df = val_df.dropna(subset=usable_features + [target_col]).copy()

    X_train = train_df[usable_features]
    y_train = train_df[target_col]
    X_val = val_df[usable_features]
    y_val = val_df[target_col]

    from itertools import product
    for max_depth, lr, n_est, col_sub in product(
        param_grid['max_depth'],
        param_grid['learning_rate'],
        param_grid['n_estimators'],
        param_grid['colsample_bytree']
    ):
        model = xgb.XGBClassifier(
            max_depth=max_depth,
            learning_rate=lr,
            n_estimators=n_est,
            colsample_bytree=col_sub,
            subsample=0.8,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1
        )
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        y_prob = model.predict_proba(X_val)[:, 1]

        # align index
        val_df_aligned = val_df.loc[y_val.index]

        best_t, best_sharpe, _ = tune_threshold_sharpe(
            y_prob, val_df_aligned, threshold_grid
        )

        if best_sharpe > best_result['sharpe']:
            best_result = {
                'max_depth': max_depth,
                'learning_rate': lr,
                'n_estimators': n_est,
                'colsample_bytree': col_sub,
                'threshold': best_t,
                'sharpe': best_sharpe,
                'model': model,
                'features': usable_features
            }

    print(f"\n{asset} BEST:")
    print({k: v for k, v in best_result.items() if k not in ('model', 'features')})

    return best_result

In [ ]:
tuned_results = {}

for asset in assets:
    tuned_results[asset] = tune_xgboost_and_threshold(
        train_data[asset],
        val_data[asset],
        asset,
        feature_cols,
        target_col
    )

In [ ]:
for asset in assets:
    print(asset,
          tuned_results[asset]['max_depth'],
          tuned_results[asset]['learning_rate'],
          tuned_results[asset]['n_estimators'],
          tuned_results[asset]['threshold'],
          tuned_results[asset]['sharpe'])

Hyperparameters controlling tree depth, learning rate, number of estimators, and feature subsampling are jointly tuned alongside the probability threshold on the validation set by maximising the annualised Sharpe ratio under transaction costs. This ensures the final configuration optimises risk-adjusted trading performance rather than classification accuracy alone.

## Backtesting

In [ ]:
import pandas as pd
from performance_utils import summarize_performance, compute_directional_metrics

def backtest_xgboost(
    test_df: pd.DataFrame,
    model,
    features: list,
    threshold: float,
    return_col: str = "fwd_ret_open_1h",
    timestamp_col: str = None,
    transaction_cost: float = 0.00035
) -> pd.DataFrame:
    """
    Backtest an XGBoost long-flat strategy using precomputed forward open-to-open returns.

    Strategy logic:
    - Predict probability of upward movement using fitted XGBoost model
    - Go long (position = 1) if probability >= threshold
    - Otherwise stay out (position = 0)
    - Use precomputed forward return column for realized next-period return
    - Apply transaction cost whenever position changes

    Parameters
    ----------
    test_df : pd.DataFrame
        Test-period data.
    model : fitted xgb.XGBClassifier
        Trained XGBoost model.
    features : list
        Feature columns used by the model.
    threshold : float
        Fixed probability threshold chosen from validation.
    return_col : str
        Column containing forward open-to-open return.
    timestamp_col : str, optional
        Optional timestamp column for readability.
    transaction_cost : float
        Proportional cost per trade (e.g. 0.00035 = 0.035%).

    Returns
    -------
    pd.DataFrame
        Backtest dataframe with probabilities, positions, costs, net returns, and equity curve.
    """
    df = test_df.copy().reset_index(drop=True)

    # Keep only rows with complete data needed for prediction and return realization
    required_cols = features + [return_col]
    if timestamp_col is not None and timestamp_col in df.columns:
        required_cols = [timestamp_col] + required_cols

    df = df.dropna(subset=required_cols).copy().reset_index(drop=True)

    # Generate predicted probabilities
    X_test = df[features]
    df["y_prob"] = model.predict_proba(X_test)[:, 1]

    # Realized next-period forward return
    df["raw_return"] = df[return_col]

    # Position: long if probability exceeds threshold, else flat
    df["position"] = (df["y_prob"] >= threshold).astype(int)

    # Position changes determine transaction costs
    df["prev_position"] = df["position"].shift(1).fillna(0)
    df["position_change"] = (df["position"] - df["prev_position"]).abs()

    # Transaction cost whenever position changes
    df["transaction_cost"] = transaction_cost * df["position_change"]

    # Gross and net strategy returns
    df["gross_return"] = df["position"] * df["raw_return"]
    df["net_return"] = df["gross_return"] - df["transaction_cost"]

    # Equity curves
    df["gross_equity_curve"] = (1 + df["gross_return"]).cumprod()
    df["net_equity_curve"] = (1 + df["net_return"]).cumprod()

    return df

In [ ]:
def run_xgboost_backtest(
    test_df: pd.DataFrame,
    model,
    features: list,
    threshold: float,
    return_col: str = "fwd_ret_open_1h",
    timestamp_col: str = None,
    transaction_cost: float = 0.00035,
    periods_per_year: int = 8760,
    risk_free_rate: float = 0.0,
    include_directional_metrics: bool = True
):
    """
    Full pipeline:
    1. Run XGBoost backtest
    2. Compute performance summary
    3. Optionally compute directional metrics
    """
    backtest_df = backtest_xgboost(
        test_df=test_df,
        model=model,
        features=features,
        threshold=threshold,
        return_col=return_col,
        timestamp_col=timestamp_col,
        transaction_cost=transaction_cost
    )

    performance_summary = summarize_performance(
        backtest_df=backtest_df,
        return_col="net_return",
        position_col="position",
        periods_per_year=periods_per_year,
        risk_free_rate=risk_free_rate
    )

    if include_directional_metrics:
        directional_summary = compute_directional_metrics(backtest_df)
        full_summary = pd.concat([performance_summary, directional_summary])
    else:
        full_summary = performance_summary

    return backtest_df, full_summary

### Results

In [ ]:
btc_test = pd.read_csv("data/split_data/BTC_test.csv")
eth_test = pd.read_csv("data/split_data/ETH_test.csv")
xrp_test = pd.read_csv("data/split_data/XRP_test.csv")
sol_test = pd.read_csv("data/split_data/SOL_test.csv")

test_datasets = {
    "BTC": btc_test,
    "ETH": eth_test,
    "XRP": xrp_test,
    "SOL": sol_test
}

all_xgb_backtest_results = {}
all_xgb_summaries = []

for asset_name, test_df in test_datasets.items():
    model = tuned_results[asset_name]["model"]
    features = tuned_results[asset_name]["features"]
    threshold = tuned_results[asset_name]["threshold"]

    backtest_results, summary = run_xgboost_backtest(
        test_df=test_df,
        model=model,
        features=features,
        threshold=threshold,
        return_col="fwd_ret_open_1h",
        timestamp_col="datetime",
        transaction_cost=0.00035,
        periods_per_year=8760,
        risk_free_rate=0.0,
        include_directional_metrics=True
    )

    all_xgb_backtest_results[asset_name] = backtest_results

    summary_df = summary.to_frame().T
    summary_df.insert(0, "Asset", asset_name)
    summary_df.insert(1, "Model", "XGBoost")
    summary_df.insert(2, "max_depth", tuned_results[asset_name]["max_depth"])
    summary_df.insert(3, "learning_rate", tuned_results[asset_name]["learning_rate"])
    summary_df.insert(4, "n_estimators", tuned_results[asset_name]["n_estimators"])
    summary_df.insert(5, "Threshold", tuned_results[asset_name]["threshold"])
    all_xgb_summaries.append(summary_df)

final_xgb_summary_table = pd.concat(all_xgb_summaries, ignore_index=True)

print("===== XGBOOST SUMMARY FOR ALL ASSETS =====")
print(final_xgb_summary_table)

In [ ]:
import os
os.makedirs("result/xgboost", exist_ok=True)

# save results
for asset_name, df in all_xgb_backtest_results.items():
    df.to_csv(f"result/xgboost/{asset_name.lower()}_backtest.csv", index=False)

final_xgb_summary_table.to_csv("result/xgboost/summary_all_assets.csv", index=False)